In [1]:
import os
import sys
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from tqdm import tqdm
import logging
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor, TQDMProgressBar
from lightning.pytorch.loggers import WandbLogger
from pytorch_lightning.utilities import rank_zero_only
from pytorch_lightning.strategies import DDPStrategy

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/dev/notebooks/simple_model_testing")
DATA_DIR = PROJECT_DIR / "data"
CHKPT_DIR = PROJECT_DIR / "checkpoints"
CHKPT_COPY_DIR = PROJECT_DIR / "checkpoints copy"
RESULT_DIR = PROJECT_DIR / "testing_results"

sys.path.append(str(PROJECT_DIR))

import models.tf_to_tg_simple as tf_to_tg_module
import models.tf_to_dna as tf_to_dna_module
import scripts.build_tf_to_tg_train_data as tf_tg_data_builder
import utils
import config
import warnings

warnings.filterwarnings(
    "ignore",
    message="You are using `torch.load` with `weights_only=False`.*",
    category=FutureWarning,
)

tf_tg_input_cache_dir = DATA_DIR / "tf_tg_training_cache"

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

In [2]:
def create_new_tf_tg_regulation_model(
    tf_bind_model_path: Path,
    tf_embeddings_tensor: torch.Tensor,
    tf_mask_tensor: torch.Tensor,
    checkpoint_path: Path | None = None,
) -> tf_to_tg_module.TFTGRegulationModel:

    # 1) Recreate the base TF→DNA model with the same hyperparameters
    base_model = tf_to_dna_module.TFPeakBindingModel(
        tf_embedding_dim=128,
        hidden_dim=128,
        dropout=0.3,
        num_layers=4,
        num_heads=4,
        dim_head=32,
    )

    # 2) Wrap in Lightning module and load checkpoint
    lit_model = tf_to_dna_module.LitTFPeakBindingModel.load_from_checkpoint(
        checkpoint_path=tf_bind_model_path,
        model=base_model,
        tf_embeddings_tensor=tf_embeddings_tensor,
        tf_mask_tensor=tf_mask_tensor,
        lr=1e-4,
        weight_decay=1e-4,
        pos_weight=None,
    )

    # 3) Get the trained base model and freeze it
    trained_tf_peak_model = lit_model.model

    trained_tf_peak_model.eval()

    for p in trained_tf_peak_model.parameters():
        p.requires_grad = False

    trained_tf_peak_model = torch.compile(
        trained_tf_peak_model,
        mode="reduce-overhead",
        fullgraph=False,
    )

    # 4) Inject into your TF→TG model
    tf_tg_model = tf_to_tg_module.TFTGRegulationModel(
        pretrained_tf_peak_model=trained_tf_peak_model,
        d_model=128,
        tf_peak_chunk_size=128,
    )

    # 5) Optionally load TF→TG checkpoint
    if checkpoint_path is not None:
        logging.info(f"Loading TF→TG model weights from checkpoint: {checkpoint_path}")

        tf_tg_ckpt = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )

        fixed = {}

        for key, value in tf_tg_ckpt["state_dict"].items():
            if key.startswith("model."):
                key = key[len("model."):]
            fixed[key] = value

        tf_tg_model.load_state_dict(fixed, strict=True)

    return tf_tg_model

def create_tf_tg_index_to_name_mappings(tf_name_to_idx, tg_id_to_idx):
    tf_idx_to_name = {idx: name for name, idx in tf_name_to_idx.items()}
    tg_idx_to_name = {idx: name for name, idx in tg_id_to_idx.items()}
    return tf_idx_to_name, tg_idx_to_name

def prepare_tftg_lookup_tables(
    peak_to_gene,
    atac_peak_map,
    atac_pseudobulk,
    rna_pseudobulk_norm,
    dataset_peaks,
    common_cells,
    max_precompute_peaks=None,
):
    valid_peak_set = set(atac_peak_map.keys())

    peak_to_gene_valid = peak_to_gene[
        peak_to_gene["peak_id"].isin(valid_peak_set)
    ].copy()

    peak_to_gene_valid["abs_dist"] = peak_to_gene_valid["TSS_dist"].abs()

    tg_to_peak_info = {}

    # Subset to only peaks within 100kb of the TG TSS and sort by distance
    for tg_norm, sub in peak_to_gene_valid.groupby("target_id_norm", sort=False):
        sub = sub[sub["abs_dist"] <= 100_000].sort_values("abs_dist")

        if sub.empty:
            continue

        # Optional cap to only use the closest N peaks per TG
        if max_precompute_peaks is not None:
            sub = sub.head(max_precompute_peaks)

        peak_ids = sub["peak_id"].tolist()
        peak_indices = np.asarray(
            [atac_peak_map[p] for p in peak_ids],
            dtype=np.int64,
        )
        peak_distances = sub["TSS_dist"].to_numpy(dtype=np.float32)

        tg_to_peak_info[tg_norm] = {
            "peak_ids": peak_ids,
            "peak_indices": peak_indices,
            "peak_distances": peak_distances,
        }

    cell_to_idx = {cell: i for i, cell in enumerate(common_cells)}

    atac_mat = (
        atac_pseudobulk
        .reindex(index=dataset_peaks, columns=common_cells)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    rna_mat = (
        rna_pseudobulk_norm
        .reindex(columns=common_cells)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    gene_to_rna_idx = {gene: i for i, gene in enumerate(rna_pseudobulk_norm.index)}

    return tg_to_peak_info, cell_to_idx, atac_mat, rna_mat, gene_to_rna_idx

def build_tftg_inputs(
    tf_tg_df,
    max_cells_per_pair=8,
    seed=123,
    silence=False,
    *,
    tg_to_peak_info,
    cell_to_idx,
    atac_mat,
    rna_mat,
    gene_to_rna_idx,
    common_cells,
    tf_name_to_idx,
    tg_id_to_idx,
    max_peaks_real,
):
    """
    Build one compact item per TF-TG edge.

    Output shapes:
        label:              [E]
        tf_idx:             [E]
        tg_idx:             [E]
        peak_indices:       [E, P]
        peak_distance:      [E, P]
        peak_mask:          [E, P]
        peak_accessibility: [E, C, P]
        tf_expression:      [E, C]
        tg_expression:      [E, C]
    """

    rng = np.random.default_rng(seed)

    tf_names = []
    tg_names = []
    cell_ids_all = []
    labels = []

    tf_indices = []
    tg_indices = []
    peak_indices_all = []
    peak_access_all = []
    peak_masks_all = []
    tf_expr_all = []
    tg_expr_all = []

    common_cells = list(common_cells)
    n_common_cells = len(common_cells)

    n_total = len(tf_tg_df)
    log_every = max(1, n_total // 50)

    for i, row in enumerate(tf_tg_df.itertuples(index=False), start=1):
        if silence == False:
            if i == 1 or i % log_every == 0 or i == n_total:
                logging.info(f"Building compact TF-TG edges: {100 * i / n_total:.1f}% ({i:,}/{n_total:,})")

        tf_name = row.tf_name
        tg_name = row.tg_id
        label = float(row.label)

        tf_idx = tf_name_to_idx.get(tf_name)
        tg_idx = tg_id_to_idx.get(tg_name)

        if tf_idx is None or tg_idx is None:
            continue

        peak_info = tg_to_peak_info.get(tg_name)
        if peak_info is None:
            continue

        peak_indices_real = list(peak_info["peak_indices"])

        n_peaks = len(peak_indices_real)
        if n_peaks == 0:
            continue

        peak_indices = np.asarray(peak_indices_real, dtype=np.int64)
        peak_mask = np.ones(n_peaks, dtype=bool)

        if n_peaks < max_peaks_real:
            pad_len = max_peaks_real - n_peaks

            peak_indices = np.pad(
                peak_indices,
                (0, pad_len),
                constant_values=0,
            )

            peak_mask = np.pad(
                peak_mask,
                (0, pad_len),
                constant_values=False,
            )

        # Sample cells
        if max_cells_per_pair is None or max_cells_per_pair >= n_common_cells:
            sampled_cells = common_cells
        else:
            sampled_cells = rng.choice(
                common_cells,
                size=max_cells_per_pair,
                replace=False,
            ).tolist()

        sampled_cell_indices = np.asarray(
            [cell_to_idx[c] for c in sampled_cells],
            dtype=np.int64,
        )

        C = len(sampled_cell_indices)
        P = max_peaks_real

        # ATAC accessibility: [C, P]
        peak_acc_matrix = np.zeros((C, P), dtype=np.float32)
        peak_acc_matrix[:, :n_peaks] = atac_mat[
            np.ix_(peak_indices_real, sampled_cell_indices)
        ].T

        # RNA expression: [C]
        tf_rna_idx = gene_to_rna_idx.get(tf_name)
        tg_rna_idx = gene_to_rna_idx.get(tg_name)

        if tf_rna_idx is None or tg_rna_idx is None:
            raise ValueError(
                f"TF or TG missing from RNA matrix after filtering: "
                f"tf_name={tf_name}, tg_name={tg_name}, "
                f"tf_rna_idx={tf_rna_idx}, tg_rna_idx={tg_rna_idx}"
            )

        tf_expr_vals = np.asarray(
            rna_mat[tf_rna_idx, sampled_cell_indices],
            dtype=np.float32,
        ).reshape(-1)

        tg_expr_vals = np.asarray(
            rna_mat[tg_rna_idx, sampled_cell_indices],
            dtype=np.float32,
        ).reshape(-1)

        # Append once per TF-TG edge
        tf_names.append(tf_name)
        tg_names.append(tg_name)
        cell_ids_all.append(sampled_cells)
        labels.append(label)

        tf_indices.append(tf_idx)
        tg_indices.append(tg_idx)
        peak_indices_all.append(peak_indices)
        peak_access_all.append(peak_acc_matrix)
        peak_masks_all.append(peak_mask)
        tf_expr_all.append(tf_expr_vals)
        tg_expr_all.append(tg_expr_vals)

    if len(labels) == 0:
        raise ValueError(
            "No TF-TG examples were created. Check TF/TG IDs, peak-to-gene mapping, "
            "and overlap with ATAC/RNA matrices."
        )

    return {
        "tf_name": tf_names,
        "tg_name": tg_names,
        "cell_ids": cell_ids_all,

        "label": torch.tensor(labels, dtype=torch.float32),

        "tf_idx": torch.tensor(tf_indices, dtype=torch.long),
        "tg_idx": torch.tensor(tg_indices, dtype=torch.long),

        "peak_indices": torch.tensor(np.stack(peak_indices_all), dtype=torch.long),
        "peak_accessibility": torch.tensor(np.stack(peak_access_all), dtype=torch.float32),
        "peak_mask": torch.tensor(np.stack(peak_masks_all), dtype=torch.bool),

        "tf_expression": torch.tensor(np.stack(tf_expr_all), dtype=torch.float32),
        "tg_expression": torch.tensor(np.stack(tg_expr_all), dtype=torch.float32),
    }

In [3]:
mm10_tf_dna_path = CHKPT_DIR / "tf_dna_mm10_3671604" / "epoch=08-val_auroc=0.9177-val_loss=0.2783.ckpt"
hg38_tf_dna_path = CHKPT_DIR / "tf_dna_hg38_3683606" / "epoch=13-val_auroc=0.9566-val_loss=0.2042.ckpt"

tf_dna_model_checkpoints = {
    "mESC": mm10_tf_dna_path,
    "iPSC": hg38_tf_dna_path,
    "Macrophage": hg38_tf_dna_path,
    "K562": hg38_tf_dna_path
}

In [ ]:
species = "hg38"
cell_type = "Macrophage"
sample_name = "buffer_1"

max_peaks_per_tg = 32
max_cells_per_pair = 20
pct_true_edges = 1
true_false_ratio = 1.0

epochs = 10
batch_size = 64

project_data_dir = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data")

if species == "mm10":
    gene_ref_file = project_data_dir / "genome_data" / "genome_annotation" / "mm10" / "Mus_musculus.GRCm39.115.gtf.gz"
elif species == "hg38":
    gene_ref_file = project_data_dir / "genome_data" / "genome_annotation" / "hg38" / "Homo_sapiens.GRCh38.113.gtf.gz"

genome_fasta_path = project_data_dir / "genome_data" / "reference_genome" / f"{species}" / f"{species}.fa"
chrom_sizes_path = project_data_dir / "genome_data" / "reference_genome" / f"{species}" / f"{species}.chrom.sizes"
chrom_sizes_path = project_data_dir / "genome_data" / "reference_genome" / f"{species}" / f"{species}.chrom.sizes"

if species == "mm10":
    train_chroms = [str(i) for i in range(1, 16)]
    val_chroms = [ str(i) for i in range(16, 18)]
    test_chroms = [str(i) for i in range(18, 20)]
elif species == "hg38":
    train_chroms = [str(i) for i in range(1, 18)]
    val_chroms = [str(i) for i in range(18, 20)]
    test_chroms = [str(i) for i in range(20, 23)]

sample_input_data_dir = PROJECT_DIR / "data" / "sample_input_data" / cell_type / sample_name

cell_type_cache_dir = DATA_DIR / f"{cell_type}_cache"

Load the ATAC, RNA, and peak distance data

In [5]:
# Read in the ATAC and RNA pseudobulk data, and the peak-to-gene distance file
atac_pseudobulk = pd.read_parquet(sample_input_data_dir / "RE_pseudobulk.parquet")
peak_to_gene_distance = pd.read_parquet(sample_input_data_dir / "peak_to_gene_dist.parquet")
rna_pseudobulk = pd.read_parquet(sample_input_data_dir / "TG_pseudobulk.parquet")

logging.info(f"ATAC peaks BEFORE peak-to-gene filtering: {atac_pseudobulk.shape[0]:,}")
# Keep only ATAC peaks that are present in the peak-to-gene distance table
valid_peak_ids = set(peak_to_gene_distance["peak_id"])

atac_pseudobulk = atac_pseudobulk.loc[
    atac_pseudobulk.index.isin(valid_peak_ids)
].copy()
logging.info(f"ATAC peaks AFTER peak-to-gene filtering: {atac_pseudobulk.shape[0]:,}")

rna_pseudobulk_norm = rna_pseudobulk.copy()
rna_pseudobulk_norm.index = rna_pseudobulk_norm.index.str.upper()

common_cells = sorted(set(rna_pseudobulk_norm.columns) & set(atac_pseudobulk.columns))

if len(common_cells) == 0:
    raise ValueError(
        "No common pseudobulk cell columns between RNA and ATAC matrices."
    )

logging.info(f"Common RNA/ATAC pseudobulk columns: {len(common_cells):,}")
    
peak_to_gene = peak_to_gene_distance.copy()
peak_to_gene["target_id_norm"] = peak_to_gene["target_id"].str.upper()


INFO - ATAC peaks BEFORE peak-to-gene filtering: 121,984
INFO - ATAC peaks AFTER peak-to-gene filtering: 118,124
INFO - Common RNA/ATAC pseudobulk columns: 792


Load merged ground truth

In [6]:
# Load and merge the ground truth files, or load from cache if already merged
merged_ground_truth_path = sample_input_data_dir / "merged_ground_truth.parquet"
if not merged_ground_truth_path.exists():
    merged_ground_truth_df = utils.load_ground_truth_files(
        config.gt_by_dataset_dict[cell_type]
    )
else:
    merged_ground_truth_df = pd.read_parquet(merged_ground_truth_path)
    
merged_ground_truth_df["Source"] = merged_ground_truth_df["Source"].str.upper()
merged_ground_truth_df["Target"] = merged_ground_truth_df["Target"].str.upper()


INFO - Loading ground truth file: chipatlas_macrophage.csv


Subset data and ground truth to the intersection between ground truth, the dataset, and TFs with embeddings

In [7]:

gt_tfs_in_rna = set(merged_ground_truth_df["Source"]).intersection(rna_pseudobulk_norm.index)
gt_tgs_in_rna = set(merged_ground_truth_df["Target"]).intersection(rna_pseudobulk_norm.index)
logging.info(f"Ground truth TFs in RNA pseudobulk: {len(gt_tfs_in_rna)} (Example: {list(gt_tfs_in_rna)[:5]})")
logging.info(f"Ground truth TGs in RNA pseudobulk: {len(gt_tgs_in_rna)} (Example: {list(gt_tgs_in_rna)[:5]})")

n_before_rna_filter = len(merged_ground_truth_df)

# Subset the ground truth to only TFs and TGs present in the rna_pseudobulk 
merged_ground_truth_df = merged_ground_truth_df[
    merged_ground_truth_df["Source"].isin(gt_tfs_in_rna) &
    merged_ground_truth_df["Target"].isin(gt_tgs_in_rna)
].copy()

logging.info(
    f"Ground truth edges after RNA TF/TG filtering: "
    f"{len(merged_ground_truth_df):,} / {n_before_rna_filter:,}"
)

# Get the map of TF name to index
tf_name_to_idx = pd.read_csv(config.tf_name_to_idx_cache_path)
tf_name_to_idx["tf_name"] = tf_name_to_idx["tf_name"].str.upper()
tf_name_to_idx = tf_name_to_idx.set_index("tf_name")["tf_idx"].to_dict()

# Only keep ground truth TFs that have embeddings (i.e. were present in the TF-DNA model training data)
gt_tfs_in_embeddings = set(tf_name_to_idx.keys()).intersection(gt_tfs_in_rna)
logging.info(f"Ground truth TFs with embeddings: {len(gt_tfs_in_embeddings)} (Example: {list(gt_tfs_in_embeddings)[:5]})")

n_before_tf_embedding_filter = len(merged_ground_truth_df)

merged_ground_truth_df = merged_ground_truth_df[
    merged_ground_truth_df["Source"].isin(gt_tfs_in_embeddings)
].copy()

logging.info(
    f"Ground truth edges after filtering to TFs with embeddings: "
    f"{len(merged_ground_truth_df):,} / {n_before_tf_embedding_filter:,}"
)

# Create a map of TG name to index for TGs present in the ground truth (and RNA pseudobulk)
tg_id_to_idx = {tg: idx for idx, tg in enumerate(merged_ground_truth_df["Target"].unique())}
    

INFO - Ground truth TFs in RNA pseudobulk: 22 (Example: ['SPI1', 'SMC1A', 'EGR1', 'EGR2', 'HIF1A'])
INFO - Ground truth TGs in RNA pseudobulk: 13594 (Example: ['GSAP', 'C8ORF44-SGK3', 'ATP8A2', 'RAB11FIP3', 'ZNF572'])
INFO - Ground truth edges after RNA TF/TG filtering: 94,801 / 223,369
INFO - Ground truth TFs with embeddings: 14 (Example: ['RELA', 'SREBF2', 'EGR2', 'SPI1', 'HIF1A'])
INFO - Ground truth edges after filtering to TFs with embeddings: 62,224 / 94,801


In [8]:
# Split genes into train/val/test based on chromosome using the GTF reference file
train_genes, val_genes, test_genes = tf_tg_data_builder.split_genes_by_chromosome(
    gene_ref_file,
    train_chroms=train_chroms,
    val_chroms=val_chroms,
    test_chroms=test_chroms
    )

# Subset the ground truth to create train/val/test splits based on the target gene chromosome splits
# (Only keeps TFs and TGs present in the ground truth and RNA pseudobulk, and only keeps TFs with embeddings)
gt_train_df, gt_val_df, gt_test_df = tf_tg_data_builder.create_train_val_test_splits(
    merged_ground_truth_df, train_genes, val_genes, test_genes
)
logging.info(f"After subsetting to TFs with embeddings and TGs in RNA pseudobulk:")
logging.info(f"  - Train interactions: {len(gt_train_df)} (TFs: {gt_train_df['Source'].nunique()}, TGs: {gt_train_df['Target'].nunique()})")
logging.info(f"  - Val interactions: {len(gt_val_df)} (TFs: {gt_val_df['Source'].nunique()}, TGs: {gt_val_df['Target'].nunique()})")
logging.info(f"  - Test interactions: {len(gt_test_df)} (TFs: {gt_test_df['Source'].nunique()}, TGs: {gt_test_df['Target'].nunique()})")

INFO - Splitting genes into train/val/test based on chromosome:
INFO - Extracted GTF attributes: ['gene_id', 'gene_version', 'gene_name', 'gene_source', 'gene_biotype', 'transcript_id', 'transcript_version', 'transcript_name', 'transcript_source', 'transcript_biotype', 'tag', 'transcript_support_level', 'exon_number', 'exon_id', 'exon_version', 'protein_id', 'protein_version', 'ccds_id']
INFO -   - Train set: 33,409 genes (chroms 1-9)
INFO -   - Validation set: 2,793 genes (chroms 18-19)
INFO -   - Test set: 2,583 genes (chroms 20-22)
INFO - Train interactions: 51412
INFO - Validation interactions: 5462
INFO - Test interactions: 3570
INFO - After subsetting to TFs with embeddings and TGs in RNA pseudobulk:
INFO -   - Train interactions: 51412 (TFs: 14, TGs: 11082)
INFO -   - Val interactions: 5462 (TFs: 14, TGs: 1203)
INFO -   - Test interactions: 3570 (TFs: 14, TGs: 799)


Create the labeled DataFrames (matches the TFs and TGs to indices for quick tensor lookup)

In [9]:
# Create labeled TF-TG datasets for train/val/test splits
# (samples true and false edges according to pct_true_edges and true_false_ratio)
tf_tg_labeled_train_df = tf_tg_data_builder._create_labeled_df(
    gt_train_df,
    pct_true_edges,
    true_false_ratio,
    seed=123,
    tf_name_to_idx=tf_name_to_idx,
    tg_id_to_idx=tg_id_to_idx,
)
tf_tg_labeled_val_df = tf_tg_data_builder._create_labeled_df(
    gt_val_df,
    pct_true_edges,
    true_false_ratio,
    seed=124,
    tf_name_to_idx=tf_name_to_idx,
    tg_id_to_idx=tg_id_to_idx,
)
tf_tg_labeled_test_df = tf_tg_data_builder._create_labeled_df(
    gt_test_df,
    pct_true_edges,
    true_false_ratio,
    seed=125,
    tf_name_to_idx=tf_name_to_idx,
    tg_id_to_idx=tg_id_to_idx,
)

tf_tg_labeled_train_df.head()

,tf_name,tg_id,label,tf_idx,tg_idx
0,CTCF,RWDD3,1.0,2,623
1,MYC,RPL7L1,0.0,120,4530
2,ATF2,DIABLO,1.0,138,8940
3,CEBPB,SEC61A1,1.0,140,2810
4,SREBF2,CNOT10,1.0,660,2411


In [10]:
tf_idx_to_name, tg_idx_to_name = create_tf_tg_index_to_name_mappings(tf_name_to_idx, tg_id_to_idx)

tf_names = tf_name_to_idx.keys()
tg_names = tg_id_to_idx.keys()

Create the peak input data for the TF-DNA model

In [11]:
# Create a map of ATAC peaks to indices in the pseudobulk matrix, filtering to valid chromosomes
dataset_peaks = atac_pseudobulk.index.to_list()

# Only use peaks from standard chromosomes (chr1-chr19 for mm10, chr1-chr22 for hg38) to avoid issues with 
# non-standard chromosomes and contigs
if config.species == "mm10":
    valid_chroms = {f"chr{i}" for i in range(1, 20)}
elif config.species == "hg38":
    valid_chroms = {f"chr{i}" for i in range(1, 23)}

dataset_peaks = [peak for peak in dataset_peaks if peak.split(":", 1)[0] in valid_chroms]
atac_peak_map = {peak: idx for idx, peak in enumerate(dataset_peaks)}

# Create the centered one-hot encoded ATAC peak array for the test set
logging.info("Creating centered one-hot encoded ATAC peak array for the test set...")
atac_peak_array = utils.create_centered_peak_onehot_array(
    peak_ids=dataset_peaks,
    genome_fasta=genome_fasta_path,
    chrom_sizes=utils.load_chrom_sizes(chrom_sizes_path),
    peak_id_to_idx=atac_peak_map,
    flank_size=128,
    dtype=np.uint8,
    pad_out_of_bounds=True,
    num_workers=8,
    show_progress=False,
    chunk_size=10000,
)
atac_peak_tensor = torch.as_tensor(atac_peak_array, dtype=torch.uint8).float()
print(f"ATAC peak tensor shape: {atac_peak_tensor.shape}")

INFO - Creating centered one-hot encoded ATAC peak array for the test set...


ATAC peak tensor shape: torch.Size([114993, 256, 4])


In [12]:

logging.info(f"Constructing TF-TG lookup tables for test set evaluation")
tg_to_peak_info, cell_to_idx, atac_mat, rna_mat, gene_to_rna_idx = prepare_tftg_lookup_tables(
    peak_to_gene=peak_to_gene,
    atac_peak_map=atac_peak_map,
    atac_pseudobulk=atac_pseudobulk,
    rna_pseudobulk_norm=rna_pseudobulk_norm,
    dataset_peaks=dataset_peaks,
    common_cells=common_cells,
    max_precompute_peaks=max_peaks_per_tg,
)
print(f"Number of TGs with at least one peak: {len(tg_to_peak_info)}")
print(f"Example TG to peak info entry: {next(iter(tg_to_peak_info.items()))}")
print(f"Number of common cells: {len(common_cells)}")
print(f"ATAC matrix shape: {atac_mat.shape}, RNA matrix shape: {rna_mat.shape}")

INFO - Constructing TF-TG lookup tables for test set evaluation


Number of TGs with at least one peak: 26281
Example TG to peak info entry: ('ABCB6', {'peak_ids': ['chr2:219217662-219218426', 'chr2:219218591-219219318', 'chr2:219211279-219212161'], 'peak_indices': array([19797, 19798, 19796]), 'peak_distances': array([   0.,  306., 1053.], dtype=float32)})
Number of common cells: 792
ATAC matrix shape: (114993, 792), RNA matrix shape: (14563, 792)


In [13]:
# Determine the maximum number of peaks to consider across all TGs in the dataset 
# to ensure consistent tensor shapes
tf_tg_df = pd.concat([tf_tg_labeled_train_df, tf_tg_labeled_val_df, tf_tg_labeled_test_df], ignore_index=True)

max_peaks_real = max(
    len(tg_to_peak_info.get(tg_name, {}).get("peak_indices", []))
    for tg_name in tf_tg_df["tg_id"]
)

# Check that at least some TGs have peaks within 100kb, otherwise the model will have no signal to learn from
n_tgs_with_peaks = sum(
    len(tg_to_peak_info.get(tg, {}).get("peak_indices", [])) > 0
    for tg in tf_tg_df["tg_id"].unique()
)
    
logging.info(f"TGs with at least one peak within 100kb: {n_tgs_with_peaks:,} / {tf_tg_df['tg_id'].nunique():,}")
logging.info(f"Max peaks per TG after filtering/capping: {max_peaks_real:,}")

INFO - TGs with at least one peak within 100kb: 12,191 / 13,084
INFO - Max peaks per TG after filtering/capping: 32


In [14]:
# Build the compact TF-TG input dataset for the test set
common_build_kwargs = dict(
    max_cells_per_pair=max_cells_per_pair,
    tg_to_peak_info=tg_to_peak_info,
    cell_to_idx=cell_to_idx,
    atac_mat=atac_mat,
    rna_mat=rna_mat,
    gene_to_rna_idx=gene_to_rna_idx,
    common_cells=common_cells,
    tf_name_to_idx=tf_name_to_idx,
    tg_id_to_idx=tg_id_to_idx,
    max_peaks_real=max_peaks_real
)

logging.info("Building TF-TG input dataset...")
tftg_inputs_train = build_tftg_inputs(
    tf_tg_labeled_train_df,
    seed=125,
    silence=False,
    **common_build_kwargs,
)

logging.info("\nBuilding validation inputs")
tftg_inputs_val = build_tftg_inputs(
    tf_tg_labeled_val_df,
    seed=124,
    **common_build_kwargs,
)

logging.info("\nBuilding test inputs")
tftg_inputs_test = build_tftg_inputs(
    tf_tg_labeled_test_df,
    seed=125,
    **common_build_kwargs,
)

INFO - Building TF-TG input dataset...
INFO - Building compact TF-TG edges: 0.0% (1/102,824)
INFO - Building compact TF-TG edges: 2.0% (2,056/102,824)
INFO - Building compact TF-TG edges: 4.0% (4,112/102,824)
INFO - Building compact TF-TG edges: 6.0% (6,168/102,824)
INFO - Building compact TF-TG edges: 8.0% (8,224/102,824)
INFO - Building compact TF-TG edges: 10.0% (10,280/102,824)
INFO - Building compact TF-TG edges: 12.0% (12,336/102,824)
INFO - Building compact TF-TG edges: 14.0% (14,392/102,824)
INFO - Building compact TF-TG edges: 16.0% (16,448/102,824)
INFO - Building compact TF-TG edges: 18.0% (18,504/102,824)
INFO - Building compact TF-TG edges: 20.0% (20,560/102,824)
INFO - Building compact TF-TG edges: 22.0% (22,616/102,824)
INFO - Building compact TF-TG edges: 24.0% (24,672/102,824)
INFO - Building compact TF-TG edges: 26.0% (26,728/102,824)
INFO - Building compact TF-TG edges: 28.0% (28,784/102,824)
INFO - Building compact TF-TG edges: 30.0% (30,840/102,824)
INFO - Building

In [15]:
import torch
from torch.utils.data import Dataset

class TFTGEdgeBagDataset(Dataset):
    def __init__(
        self,
        inputs,
        *,
        tf_embeddings_tensor,
        tf_mask_tensor,
        atac_peak_tensor,
    ):
        self.inputs = inputs
        self.tf_embeddings_tensor = tf_embeddings_tensor
        self.tf_mask_tensor = tf_mask_tensor
        self.atac_peak_tensor = atac_peak_tensor

    def __len__(self):
        return len(self.inputs["label"])

    def __getitem__(self, idx):
        tf_idx = self.inputs["tf_idx"][idx]
        tg_idx = self.inputs["tg_idx"][idx]

        peak_indices = self.inputs["peak_indices"][idx]          # [P]
        peak_sequences = self.atac_peak_tensor[peak_indices]     # [P, L, 4]

        tf_embedding = self.tf_embeddings_tensor[tf_idx]         # [T, D]
        tf_mask = self.tf_mask_tensor[tf_idx]                    # [T]

        return {
            "label": self.inputs["label"][idx],
            "tf_name": self.inputs["tf_name"][idx],
            "tg_name": self.inputs["tg_name"][idx],
            "cell_ids": self.inputs["cell_ids"][idx],
            "tf_idx": tf_idx,
            "tg_idx": tg_idx,
            "tf_embedding": tf_embedding.float(),
            "tf_mask": tf_mask.bool(),
            "peak_indices": peak_indices,
            "peak_sequences": peak_sequences,
            "peak_mask": self.inputs["peak_mask"][idx].bool(),
            "peak_accessibility": self.inputs["peak_accessibility"][idx].float(),
            "tf_expression": self.inputs["tf_expression"][idx].float(),
            "tg_expression": self.inputs["tg_expression"][idx].float(),
        }
        
def collate_tftg_edge_bags(batch):
    output = {
        "label": torch.stack([b["label"] for b in batch]).float(),

        "tf_idx": torch.stack([b["tf_idx"] for b in batch]).long(),
        "tg_idx": torch.stack([b["tg_idx"] for b in batch]).long(),

        "tf_embedding": torch.stack([b["tf_embedding"] for b in batch]),
        "tf_mask": torch.stack([b["tf_mask"] for b in batch]),

        "peak_indices": torch.stack([b["peak_indices"] for b in batch]),
        "peak_sequences": torch.stack([b["peak_sequences"] for b in batch]),
        "peak_mask": torch.stack([b["peak_mask"] for b in batch]),

        "peak_accessibility": torch.stack([b["peak_accessibility"] for b in batch]),
        "tf_expression": torch.stack([b["tf_expression"] for b in batch]),
        "tg_expression": torch.stack([b["tg_expression"] for b in batch]),

        "tf_name": [b["tf_name"] for b in batch],
        "tg_name": [b["tg_name"] for b in batch],
        "cell_ids": [b["cell_ids"] for b in batch],
    }

    E, C = output["tf_expression"].shape
    output["cell_mask"] = torch.ones(E, C, dtype=torch.bool)

    return output

In [ ]:
# Load the lookup tensors
tf_embeddings_tensor = torch.load(
    cell_type_cache_dir / "tf_embeddings.pt",
    weights_only=True,
)
tf_mask_tensor = torch.load(
    cell_type_cache_dir / "tf_masks.pt",
    weights_only=True,
)

train_dataset = TFTGEdgeBagDataset(
    tftg_inputs_train,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor,
    atac_peak_tensor=atac_peak_tensor
)

val_dataset = TFTGEdgeBagDataset(
    tftg_inputs_val,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor,
    atac_peak_tensor=atac_peak_tensor

)

test_dataset = TFTGEdgeBagDataset(
    tftg_inputs_test,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor,
    atac_peak_tensor=atac_peak_tensor
)

# Create the DataLoaders with the custom collate function for batching edge bags
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=6,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    collate_fn=collate_tftg_edge_bags,
    )

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=6,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    collate_fn=collate_tftg_edge_bags,
    )

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=6,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    collate_fn=collate_tftg_edge_bags,
    )

print(f"Created DataLoader with {len(train_dataset):,} samples and batch size {train_loader.batch_size} ({len(train_loader):,} batches)")

Created DataLoader with 96,165 samples and batch size 32 (3,006 batches)


In [17]:
tf_dna_model_chkpt = tf_dna_model_checkpoints[cell_type]

# Load the TF→TG model
tf_tg_model = create_new_tf_tg_regulation_model(
    tf_bind_model_path=tf_dna_model_chkpt,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor
)

# Generate the model predictions for the test set and create a DataFrame with TF names, TG names, and predicted scores
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
criterion = torch.nn.BCEWithLogitsLoss()

score_threshold = 0.5
pooling_mode = "lse"
pooling_temperature = 1.0

logging.info("\nStarting Lightning training...")

lit_model = tf_to_tg_module.LitTFTGRegulationModel(
    model=tf_tg_model,
    lr=1e-4,
    weight_decay=1e-4,
    pos_weight=None,
    pooling_mode=pooling_mode,
    pooling_temperature=pooling_temperature,
    enable_timing_sync=False,
)

output_dir = PROJECT_DIR / "data" / "checkpoints" / "simplified_model" / f"{cell_type}_{sample_name}_simplified_model_test"

checkpoint_callback = ModelCheckpoint(
    dirpath=output_dir,
    filename="epoch={epoch:02d}-val_auroc={val/auroc:.4f}-val_loss={val/loss:.4f}",
    monitor="val/auroc",
    mode="max",
    save_top_k=500,
    save_last=True,
    auto_insert_metric_name=False,
)

early_stopping_callback = EarlyStopping(
    monitor="val/loss",
    mode="min",
    patience=15,
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

wandb_logger = WandbLogger(
    project="tf_tg_regulation_prediction",
    name="Simplified_TF_TG_Model_Test",
    save_dir=output_dir,
)

wandb_logger.log_hyperparams({
    "sample_name": sample_name,
    "epochs": epochs,
    "batch_size": batch_size,
    "num_batches": len(train_loader),
    "num_gpus": 1,
    "num_nodes": 1,
    "run_name": "Simplified_TF_TG_Model_Test",
    "sample_pairs": len(train_dataset),
    "max_peaks_per_tg": max_peaks_per_tg,
    "max_cells_per_pair": max_cells_per_pair,
    "pct_true_edges": pct_true_edges,
    "true_false_ratio": true_false_ratio,
    "pooling_mode": pooling_mode,
    "pooling_temperature": pooling_temperature,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "flank_size": 128,
    "max_precompute_peaks": max_peaks_per_tg,
    "persistent_workers": True,
    "tf_bind_model_path": str(tf_dna_model_chkpt),
})

world_size = int(
    os.environ.get(
        "WORLD_SIZE",
        os.environ.get("SLURM_NTASKS", "1"),
    )
)

use_ddp = world_size > 1

logging.info(f"Num GPUs: {world_size} | Batch size: {batch_size}")
logging.info(f"Num steps per epoch: {len(train_loader)}")

strategy=DDPStrategy(
    process_group_backend="nccl",
    find_unused_parameters=False,
) if use_ddp else "auto"

trainer = pl.Trainer(
    max_epochs=epochs,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    num_nodes=1,
    strategy=strategy,
    precision="16-mixed",
    logger=wandb_logger,
    gradient_clip_val=1.0,
    gradient_clip_algorithm="norm",
    log_every_n_steps=10,
    default_root_dir=output_dir,
    enable_progress_bar=True,
    enable_checkpointing=True,
    check_val_every_n_epoch=1,
)

INFO - 
Starting Lightning training...
INFO - Num GPUs: 1 | Batch size: 32
INFO - Num steps per epoch: 3006
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python ...
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [19]:
import importlib
importlib.reload(tf_to_tg_module)

<module 'models.tf_to_tg_simple' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/dev/notebooks/simple_model_testing/models/tf_to_tg_simple.py'>

In [20]:
trainer.fit(
    lit_model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /gpfs/Home/esm5360/.netrc.
wandb: Currently logged in as: luminarada (luminarada-penn-state-health) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ TFTGRegulationModel │  1.9 M │ train │     0 │
│ 1 │ train_acc │ BinaryAccuracy      │      0 │ train │     0 │
│ 2 │ val_acc   │ BinaryAccuracy      │      0 │ train │     0 │
└───┴───────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 174 K                                                                                            
Non-trainable params: 1.7 M                                                                                        
Total params: 1.9 M                                                                                                
Total estimated model params size (MB): 7                                                                          
Modules in train mode: 31                                                                                          
Modules in eval mode: 127                                                                                          
Total FLOPs: 0

Output()

...

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/pytorch_lightning/utilities/data.py:79: 
Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 5. To avoid any 
miscalculations, use `self.log(..., batch_size=batch_size)`.

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/pytorch_lightning/utilities/data.py:79: 
Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 6. To avoid any 
miscalculations, use `self.log(..., batch_size=batch_size)`.


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
